# 01 — Exploratory Data Analysis (EDA)

**Project:** MentalBERT-CSSR  
**Goal of this notebook:** Characterise the CSSR-S Reddit corpus before any cleaning or modelling.

---

## Scope

| Item | Decision |
|------|----------|
| Text field | `content` |
| Target | `severity` (human annotations, labels **0–6**) |
| Ignored for training | `gpt_label`, `claude_label`, `gemini_label`, `llama_label`, `mistral_label` |
| Labels | **Never modified** |
| Encoder | Not used here — tokenization is Notebook 3 |

## Design decisions (why this notebook looks this way)

1. **Config-driven paths & hyperparameters** — all knobs live in `configs/default.yaml` so later experiments stay comparable.
2. **Shared `utils/` package** — EDA logic is modular (`utils/eda.py`, `utils/io.py`) to avoid copy-paste across notebooks and to keep this notebook readable for a paper appendix.
3. **Whitespace token proxy only** — character/word counts are safe for EDA. True MentalBERT subword lengths (and `max_length` recommendation) are deferred to Notebook 3, where the official tokenizer is available.
4. **No preprocessing here** — we inspect *raw* text. Cleaning that preserves clinical signal (negations, affect words, meaningful punctuation) happens in Notebook 2.
5. **Artifacts are first-class** — every plot and summary table is written under `RESULTS/` for reproducibility and portfolio evidence.

> **Ethics.** Posts may contain distressing content. This analysis is research-only and is not a clinical tool.

## 1. Environment, configuration, and reproducibility

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ---------------------------------------------------------------------------
# Ensure project root is on sys.path whether the kernel CWD is PROJECT/ or
# PROJECT/NOTEBOOKS/. This keeps imports stable in Jupyter / VS Code / Cursor.
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent]
PROJECT_ROOT = None
for candidate in CANDIDATES:
    if (candidate / "configs" / "default.yaml").exists() and (candidate / "utils").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate project root (expected configs/default.yaml and utils/). "
        "Open the notebook from the MentalBERT-CSSR project directory."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import ensure_directories, get_logger, load_config, set_seed
from utils.eda import (
    BENCHMARK_SAFE_IGNORE,
    add_text_length_features,
    build_eda_report,
    class_distribution,
    length_by_class,
    length_statistics_table,
    plot_imbalance_ratio,
    plot_length_boxplot_by_class,
    plot_length_histogram,
    plot_severity_distribution,
    summarize_duplicates,
    summarize_missing,
)
from utils.io import load_raw_dataset, training_columns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

cfg = load_config()
set_seed(cfg.SEED)
ensure_directories(cfg)
logger = get_logger("notebook.01_eda")

TEXT_COL = cfg.data.text_column
LABEL_COL = cfg.data.label_column
PLOTS_DIR = Path(cfg.paths.plots_path)
METRICS_DIR = Path(cfg.paths.metrics_path)
DPI = int(cfg.eda.figure_dpi)
PERCENTILES = list(cfg.eda.length_percentiles)

logger.info("Project root : %s", PROJECT_ROOT)
logger.info("Raw data     : %s", cfg.paths.raw_data)
logger.info("Seed         : %s", cfg.SEED)
logger.info("Text / label : %s / %s", TEXT_COL, LABEL_COL)
logger.info("Ignore cols  : %s", list(cfg.data.ignore_columns))

## 2. Load raw dataset

We load the **immutable** raw CSV. Schema checks ensure `content` and `severity` exist.  
LLM label columns are retained in the raw frame for inspection of missingness, but they are **excluded** from the training-oriented view.

In [ ]:
raw_path = Path(cfg.paths.raw_data)
df_raw = load_raw_dataset(
    path=raw_path,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
)

print("=" * 72)
print("DATASET SHAPE")
print("=" * 72)
print(f"rows={df_raw.shape[0]:,}  columns={df_raw.shape[1]}")
print(f"memory_usage_mb={df_raw.memory_usage(deep=True).sum() / (1024 ** 2):.3f}")
print()
print("Columns:", list(df_raw.columns))

## 3. Head / tail / random sample

Quick qualitative inspection of post content and severity labels.  
Sensitive text is displayed only for research inspection inside this controlled notebook.

In [ ]:
display_cols = [c for c in [TEXT_COL, LABEL_COL, "url", "author", "created"] if c in df_raw.columns]

print("HEAD")
display(df_raw[display_cols].head(5))

print("TAIL")
display(df_raw[display_cols].tail(5))

print("RANDOM SAMPLE (seeded)")
display(df_raw[display_cols].sample(n=min(cfg.eda.random_sample_rows, len(df_raw)), random_state=cfg.SEED))

## 4. Column types, missing values, and duplicates

In [ ]:
print("=" * 72)
print("COLUMN DTYPES")
print("=" * 72)
dtype_table = pd.DataFrame(
    {
        "column": df_raw.columns,
        "dtype": [str(t) for t in df_raw.dtypes],
        "n_unique": [df_raw[c].nunique(dropna=False) for c in df_raw.columns],
        "n_non_null": [df_raw[c].notna().sum() for c in df_raw.columns],
    }
)
display(dtype_table)

print("MISSING VALUES")
missing_table = summarize_missing(df_raw)
display(missing_table)

print("DUPLICATES")
dup_summary = summarize_duplicates(df_raw, text_column=TEXT_COL)
display(pd.DataFrame([dup_summary]))

# Persist tables for the paper / portfolio artefact set
dtype_table.to_csv(METRICS_DIR / "eda_dtypes.csv", index=False)
missing_table.to_csv(METRICS_DIR / "eda_missing_values.csv", index=False)
pd.DataFrame([dup_summary]).to_csv(METRICS_DIR / "eda_duplicates.csv", index=False)
logger.info("Wrote dtype / missing / duplicate summaries to %s", METRICS_DIR)

In [ ]:
# Interpret missingness on LLM benchmark columns (NOT training targets)
llm_cols = [c for c in BENCHMARK_SAFE_IGNORE if c in df_raw.columns]
if llm_cols:
    llm_missing = missing_table[missing_table["column"].isin(llm_cols)].copy()
    print("LLM benchmark columns — missingness (ignored during training):")
    display(llm_missing)
    assert df_raw[TEXT_COL].isna().sum() == 0, "Unexpected nulls in content"
    assert df_raw[LABEL_COL].isna().sum() == 0, "Unexpected nulls in severity"
    print("Training-critical fields (content, severity): zero missing values.")

## 5. Training-oriented column view

Drop LLM label columns from the working frame used for severity analysis.  
Raw file on disk is untouched.

In [ ]:
df = training_columns(
    df_raw,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
    ignore_columns=list(cfg.data.ignore_columns),
)

print("Working frame columns (LLM labels excluded):", list(df.columns))
print("Shape:", df.shape)
display(df.head(3))

## 6. Severity distribution and class imbalance

C-SSRS severity is a **7-class** ordinal/categorical target. Imbalance will matter for:

- stratified train/val/test splits (Notebook 4),
- macro-averaged metrics (primary for rare high-severity classes),
- optional class weights (future experiment — not applied yet).

In [ ]:
print("Unique severity labels:", sorted(df[LABEL_COL].dropna().unique().tolist()))
print("Expected labels      :", list(range(cfg.NUM_LABELS)))

dist = class_distribution(df, label_column=LABEL_COL)
display(dist)

majority = int(dist.loc[dist["count"].idxmax(), "severity"])
minority = int(dist.loc[dist["count"].idxmin(), "severity"])
print(f"Majority class: {majority} ({int(dist['count'].max())} posts)")
print(f"Minority class: {minority} ({int(dist['count'].min())} posts)")
print(f"Max imbalance ratio (majority/minority): {dist['imbalance_ratio_vs_majority'].max():.3f}")

dist.to_csv(METRICS_DIR / "eda_class_distribution.csv", index=False)

plot_severity_distribution(
    dist,
    output_path=PLOTS_DIR / "eda_severity_distribution.png",
    dpi=DPI,
)
plot_imbalance_ratio(
    dist,
    output_path=PLOTS_DIR / "eda_class_imbalance_ratio.png",
    dpi=DPI,
)

## 7. Text length analysis

We compute:

- **character count**
- **word count** (whitespace split)
- **approximate token count** (same as word count here — explicit proxy)

Statistics include min / mean / max / median / std and configured percentiles (default: 50, 75, 90, **95**, **99**).

**Why not MentalBERT tokens yet?**  
Loading the domain tokenizer pulls model cards / vocab and belongs with truncation analysis (`truncation_side="left"`) in Notebook 3. Mixing that here would couple EDA to GPU/network availability.

In [ ]:
df_len = add_text_length_features(df, text_column=TEXT_COL)

length_cols = ["char_count", "word_count", "token_count_approx"]
length_stats = length_statistics_table(df_len, length_cols, percentiles=PERCENTILES)

print("GLOBAL LENGTH STATISTICS")
display(length_stats.round(3))

print("KEY WORD-COUNT SUMMARY")
wc = df_len["word_count"]
print(f"  average words : {wc.mean():.3f}")
print(f"  minimum words : {int(wc.min())}")
print(f"  maximum words : {int(wc.max())}")
print(f"  95 percentile : {wc.quantile(0.95):.3f}")
print(f"  99 percentile : {wc.quantile(0.99):.3f}")

length_stats.to_csv(METRICS_DIR / "eda_length_statistics.csv", index=False)

In [ ]:
print("WORD COUNT BY SEVERITY")
wc_by_sev = length_by_class(df_len, length_column="word_count", label_column=LABEL_COL)
display(wc_by_sev.round(3))
wc_by_sev.to_csv(METRICS_DIR / "eda_word_count_by_severity.csv", index=False)

print("CHAR COUNT BY SEVERITY")
cc_by_sev = length_by_class(df_len, length_column="char_count", label_column=LABEL_COL)
display(cc_by_sev.round(3))
cc_by_sev.to_csv(METRICS_DIR / "eda_char_count_by_severity.csv", index=False)

## 8. Histograms and boxplots

Histograms show global length shape; boxplots expose per-class outliers that could affect padding / truncation later.

In [ ]:
plot_length_histogram(
    df_len["word_count"],
    output_path=PLOTS_DIR / "eda_word_count_histogram.png",
    xlabel="Word count (whitespace tokens)",
    title="Distribution of Post Word Counts",
    dpi=DPI,
)

plot_length_histogram(
    df_len["char_count"],
    output_path=PLOTS_DIR / "eda_char_count_histogram.png",
    xlabel="Character count",
    title="Distribution of Post Character Counts",
    dpi=DPI,
)

plot_length_histogram(
    df_len["token_count_approx"],
    output_path=PLOTS_DIR / "eda_token_count_approx_histogram.png",
    xlabel="Approximate token count (whitespace proxy)",
    title="Distribution of Approximate Token Counts (EDA Proxy)",
    dpi=DPI,
)

plot_length_boxplot_by_class(
    df_len,
    length_column="word_count",
    label_column=LABEL_COL,
    output_path=PLOTS_DIR / "eda_word_count_boxplot_by_severity.png",
    ylabel="Word count",
    title="Word Count by Severity Class",
    dpi=DPI,
)

plot_length_boxplot_by_class(
    df_len,
    length_column="char_count",
    label_column=LABEL_COL,
    output_path=PLOTS_DIR / "eda_char_count_boxplot_by_severity.png",
    ylabel="Character count",
    title="Character Count by Severity Class",
    dpi=DPI,
)

print("Figures written to:", PLOTS_DIR)

## 9. Additional qualitative checks

Edge cases (very short / very long posts) help Notebook 2 decide what is "invalid" vs. clinically meaningful brevity.

In [ ]:
print("SHORTEST POSTS (by word count)")
display(
    df_len.nsmallest(5, "word_count")[[TEXT_COL, LABEL_COL, "word_count", "char_count"]]
)

print("LONGEST POSTS (by word count)")
display(
    df_len.nlargest(5, "word_count")[[TEXT_COL, LABEL_COL, "word_count", "char_count"]]
)

# Empty / whitespace-only content check (should be rare / zero)
blank_mask = df_len[TEXT_COL].fillna("").astype(str).str.strip().eq("")
print(f"Blank / whitespace-only posts: {int(blank_mask.sum())}")

## 10. Persist master EDA report + experiment stamp

A JSON report captures the numerical EDA snapshot. An append-only JSONL experiment log records that this EDA run occurred with the active config (timestamp + hyperparameters).

In [ ]:
eda_report = build_eda_report(
    df=df_raw,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
    percentiles=PERCENTILES,
)

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
eda_report["timestamp_utc"] = timestamp
eda_report["dataset_path"] = str(raw_path)
eda_report["dataset_name"] = cfg.experiment.dataset_name
eda_report["seed"] = cfg.SEED

report_path = METRICS_DIR / "eda_report.json"
with report_path.open("w", encoding="utf-8") as fh:
    json.dump(eda_report, fh, indent=2, ensure_ascii=False)

experiment_record = {
    "timestamp_utc": timestamp,
    "stage": "eda",
    "notebook": "01_EDA.ipynb",
    "dataset_name": cfg.experiment.dataset_name,
    "dataset_path": str(raw_path),
    "n_rows": int(df_raw.shape[0]),
    "n_columns": int(df_raw.shape[1]),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "num_labels": cfg.NUM_LABELS,
    "model_name": cfg.MODEL_NAME,
    "seed": cfg.SEED,
    "hyperparameters": {
        "learning_rate": cfg.LEARNING_RATE,
        "batch_size": cfg.BATCH_SIZE,
        "epochs": cfg.EPOCHS,
        "dropout": cfg.DROPOUT,
        "weight_decay": cfg.WEIGHT_DECAY,
        "warmup_ratio": cfg.WARMUP_RATIO,
        "label_smoothing": cfg.LABEL_SMOOTHING,
        "max_length": cfg.MAX_LENGTH,
        "optimizer": cfg.training.optimizer,
        "scheduler": cfg.training.scheduler,
    },
    "artifacts": {
        "eda_report": str(report_path),
        "plots_dir": str(PLOTS_DIR),
        "metrics_dir": str(METRICS_DIR),
    },
}

log_path = Path(cfg.paths.experiment_log)
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("a", encoding="utf-8") as fh:
    fh.write(json.dumps(experiment_record, ensure_ascii=False) + "\n")

logger.info("EDA report → %s", report_path)
logger.info("Experiment log append → %s", log_path)
print("EDA complete.")
print(f"Report : {report_path}")
print(f"Plots  : {PLOTS_DIR}")
print(f"Metrics: {METRICS_DIR}")

## 11. Findings checklist (fill after you run the notebook)

Use this section as a paper-ready summary once cells above have executed:

| Question | Observation (after run) |
|----------|-------------------------|
| How many posts? | _see shape cell_ |
| Missing `content` / `severity`? | _expect 0 / 0_ |
| Exact duplicate rows? | _see duplicates summary_ |
| Most frequent severity? | _see class distribution_ |
| Rarest severity? | _see class distribution_ |
| Mean / p95 / p99 word count? | _see length statistics_ |
| Any blank posts? | _see qualitative checks_ |

### Implications for later notebooks

- **Notebook 2 (Preprocessing):** Remove duplicates / nulls / invalid characters only; preserve negations, emotion lexicon, and meaning-bearing punctuation.
- **Notebook 3 (Tokenization):** Replace the whitespace token proxy with MentalBERT subword lengths; set `truncation_side="left"` and recommend `max_length` from p95/p99.
- **Notebook 4 (Training):** Stratify splits; emphasise **macro-F1** because minority high-severity classes are clinically critical.

---

**Stop here.** Do not proceed to Notebook 2 until this EDA notebook is reviewed and approved.